# Predicting Electric Vehicle Purchase Intent

**Project Overview:**
The goal of this project is to identify the primary drivers behind consumer intent to purchase an Electric Vehicle (EV) and build a robust predictive model. Using a dataset of over 668,000 respondents, the aim is to cut through statistical noise to find the true predictors of EV adoption.

**Methodology:**
1. **Statistical Exploratory Data Analysis (EDA):** Using effect sizes (Cramér's V and Cohen's d) to find the practical significance of each feature and bypass sample-size inflation.
2. **Predictive Modeling:** Training Gradient Boosting models (LightGBM) to validate our statistical findings and build a lean, production-ready classifier.

In [1]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('train.csv')
data.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


### 1. Initial Data Inspection
Check first the dimensions of the dataset, verify if there are any missing values, and review the basic summary statistics.

In [3]:
data.shape

(668665, 15)

In [4]:
data.isna().sum()

id                             0
Age                            0
Annual_Income_USD              0
Daily_Commute_km               0
Number_of_Cars_Owned           0
Charging_Stations_Near_Home    0
Charging_Stations_Near_Work    0
Environmental_Concern_Level    0
Gender                         0
City_Type                      0
Current_Car_Type               0
Home_Charging_Possible         0
Subsidy_Available              0
Range_Anxiety_Level            0
Will_Buy_EV                    0
dtype: int64

In [5]:
data.describe()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level
count,668665.00000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000
mean,334332.00000,47.039171,84769.266989,32.158298,1.712626,4.960408,7.176314,2.935477
std,193027.10321,12.875448,28648.029042,18.730474,0.729275,3.926843,5.186627,1.429119
min,0.00000,25.000000,30000.000000,5.000000,1.000000,0.000000,0.000000,1.000000
25%,167166.00000,36.000000,67376.000000,17.200000,1.000000,2.000000,3.000000,2.000000
50%,334332.00000,47.000000,84880.000000,33.600000,2.000000,4.000000,6.000000,3.000000
75%,501498.00000,58.000000,102753.000000,47.400000,2.000000,7.000000,10.000000,4.000000
max,668664.00000,69.000000,188549.000000,98.700000,4.000000,14.000000,19.000000,5.000000


In [6]:
data.columns.to_list()

['id',
 'Age',
 'Annual_Income_USD',
 'Daily_Commute_km',
 'Number_of_Cars_Owned',
 'Charging_Stations_Near_Home',
 'Charging_Stations_Near_Work',
 'Environmental_Concern_Level',
 'Gender',
 'City_Type',
 'Current_Car_Type',
 'Home_Charging_Possible',
 'Subsidy_Available',
 'Range_Anxiety_Level',
 'Will_Buy_EV']

### 2. Target Variable Definition & Base Rate
Before diving into feature analysis, define first the target variable (`Will_Buy_EV`). It will be encoded as a binary variable (`1` for 'Yes', `0` for 'No') and calculate the overall base rate.

Establishing the base rate is a critical first step. In imbalanced datasets, knowing the prior probability (the base rate) prevents us from being misled by standard accuracy metrics later on.

In [7]:
# encode target
data['target'] = (data['Will_Buy_EV'] == 'Yes').astype(int)

base_rate = data['target'].mean()

print(f'P(Will_Buy_EV = Yes): {base_rate:.4f}')
print(f'P(Will_Buy_EV = No): {(1-base_rate):.4f}')

P(Will_Buy_EV = Yes): 0.1746
P(Will_Buy_EV = No): 0.8254


### 3. Categorical Drivers (Noise vs. Signal)

With a massive dataset ($N = 668,665$), standard p-values from Chi-Square tests will almost always appear statistically significant, even for trivial differences. This is known as sample-size inflation.

To combat this, one can use **Cramér's V**. This metric normalizes the association strength on a scale from 0 to 1, allowing us to see which categorical features actually drive purchase intent and which are simply noise. We will also compute the conditional probability of buying an EV given each category.

In [8]:
# categorical features

categorical_columns = ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
                'Subsidy_Available', 'Range_Anxiety_Level']

from scipy import stats

for col in categorical_columns:
    contingency = pd.crosstab(data[col], data['target'])
    chi2, p, dof, ex = stats.chi2_contingency(contingency)
    n = contingency.sum().sum()
    cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))

    print(f'\nFeature: {col} | Cramer\'s V: {cramers_v:.4f} ')
    rates = data.groupby(col)['target'].agg(['count', 'mean']).rename(columns={'mean': 'P(Buy_EV)'})
    print(rates.to_string())


Feature: Gender | Cramer's V: 0.0070 
         count  P(Buy_EV)
Gender                   
Female  295427   0.177641
Male    367954   0.172253
Other     5284   0.173732

Feature: City_Type | Cramer's V: 0.0333 
            count  P(Buy_EV)
City_Type                   
Rural      123983   0.193389
Suburban   255377   0.180936
Urban      289305   0.161058

Feature: Current_Car_Type | Cramer's V: 0.0161 
                   count  P(Buy_EV)
Current_Car_Type                   
Hatchback          79438   0.174299
SUV               246545   0.180953
Sedan             303459   0.171967
Truck              39223   0.156413

Feature: Home_Charging_Possible | Cramer's V: 0.0836 
                         count  P(Buy_EV)
Home_Charging_Possible                   
No                      205988   0.127085
Yes                     462677   0.195819

Feature: Subsidy_Available | Cramer's V: 0.3424 
                    count  P(Buy_EV)
Subsidy_Available                   
No                 248756   0.00

### 4. Numerical Drivers and Effect Size

Just as with categorical features, we need a scale-independent way to measure how much our numerical features separate EV buyers from non-buyers. We will use three metrics:
* **Group Means:** The raw averages for buyers vs. non-buyers.
* **Point-Biserial Correlation ($r_{pb}$):** Measures the linear association between a continuous feature and a binary target.
* **Cohen's $d$:** A standardized effect size that measures the distance between the two groups in terms of standard deviations. A larger Cohen's $d$ (e.g., > 0.5) indicates a meaningful real-world difference, regardless of the feature's original unit of measurement.

In [9]:
# numerical features

numerical_columns = ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 
            'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']

num_summary = []
for col in numerical_columns:
    group0 = data[data['target'] == 0][col]
    group1 = data[data['target'] == 1][col]

    # point-biserial correlation
    r_pb, p_val = stats.pointbiserialr(data[col], data['target'])

    # cohen's d
    n0, n1 = len(group0), len(group1)
    s0, s1 = group0.std(), group1.std()
    s_pooled = np.sqrt(((n0 - 1)*s0**2 + (n1 - 1)*s1**2) / (n0 + n1 - 2))
    d = (group1.mean() - group0.mean()) / s_pooled

    num_summary.append({'Feature': col, 'Mean_No': group0.mean(), 'Mean_Yes': group1.mean(), 'Point_Biserial_r': r_pb, 'Cohens_d': d})

num_data = pd.DataFrame(num_summary).sort_values(by='Cohens_d', key=abs, ascending=False)
print(num_data.to_string(index=False))

                    Feature      Mean_No     Mean_Yes  Point_Biserial_r  Cohens_d
Environmental_Concern_Level     2.630395     4.377268          0.464079  1.379940
          Annual_Income_USD 81794.588556 98827.302948          0.225729  0.610302
           Daily_Commute_km    32.553478    30.290714         -0.045866 -0.120934
Charging_Stations_Near_Home     4.989947     4.820807         -0.016353 -0.043079
Charging_Stations_Near_Work     7.205490     7.038432         -0.012229 -0.032212
                        Age    47.083293    46.830654         -0.007450 -0.019622
       Number_of_Cars_Owned     1.711319     1.718802          0.003896  0.010261


### 5. Statistical Determinants of EV Purchase Intent

#### Categorical Feature Effects
| Feature | Cramér's V | Association Strength | Conditional Purchase Rates |
| :--- | :--- | :--- | :--- |
| **Subsidy_Available** | **0.3424** | **Very Strong** | No: 0.58% <br>Yes: 27.47% |
| **Range_Anxiety_Level** | **0.1159** | **Moderate** | Low: 18.90% <br>Medium: 4.17% <br>High: 0.14% |
| **Home_Charging_Possible** | **0.0836** | **Weak/Moderate**| No: 12.71% <br>Yes: 19.58% |
| **City_Type** | **0.0333** | **Weak** | Rural: 19.34% <br>Suburban: 18.09% <br>Urban: 16.11% |
| **Current_Car_Type** | **0.0161** | **Negligible** | SUV: 18.10% <br>Sedan: 17.20% |
| **Gender** | **0.0070** | **None** | Female: 17.76% <br>Male: 17.23% |

* `Subsidy_Available` exerts the strongest categorical effect. The presence of a government subsidy expands adoption intent nearly fiftyfold, highlighting that initial capital expenditure remains the primary structural bottleneck.
* `Range_Anxiety_Level` serves as a severe deterrent. High anxiety suppresses the purchase rate to a negligible 0.14%.
* Conventional demographic vectors (`Gender`, `Current_Car_Type`) exhibit near-zero association with purchase intent, confirming that EV market penetration cuts uniformly across gender lines and existing vehicle classes.

---

#### Numerical Feature Effects
| Feature | Mean (Non-Buyers) | Mean (Buyers) | Point-Biserial $r$ | Cohen's $d$ |
| :--- | :--- | :--- | :--- | :--- |
| **Environmental_Concern_Level** | 2.63 | 4.38 | 0.4641 | **1.3799** |
| **Annual_Income_USD** | $81,794 | $98,827 | 0.2257 | **0.6103** |
| **Daily_Commute_km** | 32.55 | 30.29 | -0.0459 | **-0.1209** |
| **Charging_Stations_Near_Home** | 4.99 | 4.82 | -0.0164 | **-0.0431** |
| **Charging_Stations_Near_Work** | 7.21 | 7.04 | -0.0122 | **-0.0322** |
| **Age** | 47.08 | 46.83 | -0.0075 | **-0.0196** |
| **Number_of_Cars_Owned** | 1.71 | 1.72 | 0.0039 | **0.0103** |

* `Environmental_Concern_Level` represents the single most powerful differentiator in the dataset ($d = 1.38$). Buyers demonstrate profound eco-consciousness (mean score 4.38/5) compared to neutral non-buyers (2.63/5).
* `Annual_Income_USD` confirms a moderate-to-strong financial barrier ($d = 0.61$). Prospective buyers out-earn non-buyers by an average of $17,000 annually.
* Macro-level geographic counters (like station counts) and demographics (`Age`, `Cars_Owned`) yield trivial effect sizes. Consumers are indifferent to the raw volume of public charging stations; personal home infrastructure and psychological comfort govern intent instead.

### 6. Machine Learning Model

Our statistical analysis revealed that a few features (like `Subsidy_Available` and `Environmental_Concern_Level`) are massive drivers, while others (like `Gender` and `Age`) are statistical dead weight.

To prove this in a predictive context, we will build a **LightGBM** classifier. Gradient Boosting Decision Trees (GBDTs) are highly efficient for tabular data with mixed feature types and inherently handle non-linear relationships.

We will evaluate and compare two models using **Stratified 5-Fold Cross-Validation**:
* **Option A (Pruned Signal Only):** Uses only the top 6 features identified in our EDA.
* **Option B (All Features):** Uses all 13 available features.

By tracking the **Out-Of-Fold (OOF) ROC-AUC**, we can verify if the extra demographic and geographic features actually improve the model's predictive power, or if they simply add unnecessary complexity.

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

warnings.filterwarnings('ignore')

# define the target and encode categoricals for LightGBM
y = data['target'].values

cat_cols = ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']
for col in cat_cols:
    if col in data.columns:
        data[col] = data[col].astype('category')

# defien the feature sets
strong_features = [
    'Daily_Commute_km', 'Annual_Income_USD', 'Environmental_Concern_Level',
    'Home_Charging_Possible', 'Range_Anxiety_Level', 'Subsidy_Available'
]

all_features = strong_features + [
    'Age', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 
    'Charging_Stations_Near_Work', 'Gender', 'City_Type', 'Current_Car_Type'
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# evaluation function
def evaluate_lgb(features, name, colsample=1.0):
    print(f"\n\nEvaluating {name} ({len(features)} features)")
    print(f"Features: {features}")
    
    X = data[features].copy()
    oof_preds = np.zeros(len(data))
    fold_aucs = []
    
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'colsample_bytree': colsample,
        'subsample': 0.8,
        'n_estimators': 1000,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    start_t = time.time()
    feature_importances = np.zeros(len(features))
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        X_tr, y_tr = X.iloc[train_idx], y[train_idx]
        X_va, y_va = X.iloc[val_idx], y[val_idx]
        
        clf = lgb.LGBMClassifier(**params)
        clf.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(50, verbose=False)]
        )
        
        val_probs = clf.predict_proba(X_va)[:, 1]
        oof_preds[val_idx] = val_probs
        auc = roc_auc_score(y_va, val_probs)
        fold_aucs.append(auc)
        feature_importances += clf.feature_importances_ / 5.0
        print(f"  Fold {fold+1} ROC-AUC: {auc:.5f} (Best Iteration: {clf.best_iteration_})")
        
    total_time = time.time() - start_t
    overall_auc = roc_auc_score(y, oof_preds)
    print(f"*** {name} Overall OOF ROC-AUC: {overall_auc:.5f} (Mean: {np.mean(fold_aucs):.5f} +/- {np.std(fold_aucs):.5f})")
    print(f"*** Total Training Time: {total_time:.1f}s")
    
    fi_df = pd.DataFrame({'Feature': features, 'Importance': feature_importances}).sort_values('Importance', ascending=False)
    print("Feature Importances:")
    print(fi_df.to_string(index=False))
    
    return overall_auc, oof_preds, fi_df

# execute the comparison
auc_a, oof_a, fi_a = evaluate_lgb(strong_features, "Option A (Pruned Signal Only)", colsample=1.0)
auc_b, oof_b, fi_b = evaluate_lgb(all_features, "Option B (All Features with Subsampling 0.8)", colsample=0.8)

print("FINAL COMPARISON:")
print(f"Option A (6 Strong Features): OOF ROC-AUC = {auc_a:.5f}")
print(f"Option B (All 13 Features):   OOF ROC-AUC = {auc_b:.5f}")
print(f"Delta (Option B - Option A):  {auc_b - auc_a:+.5f}")



Evaluating Option A (Pruned Signal Only) (6 features)
Features: ['Daily_Commute_km', 'Annual_Income_USD', 'Environmental_Concern_Level', 'Home_Charging_Possible', 'Range_Anxiety_Level', 'Subsidy_Available']
  Fold 1 ROC-AUC: 0.93988 (Best Iteration: 349)
  Fold 2 ROC-AUC: 0.94092 (Best Iteration: 273)
  Fold 3 ROC-AUC: 0.94227 (Best Iteration: 424)
  Fold 4 ROC-AUC: 0.94182 (Best Iteration: 325)
  Fold 5 ROC-AUC: 0.94131 (Best Iteration: 484)
*** Option A (Pruned Signal Only) Overall OOF ROC-AUC: 0.94122 (Mean: 0.94124 +/- 0.00082)
*** Total Training Time: 66.0s
Feature Importances:
                    Feature  Importance
          Annual_Income_USD      5098.0
           Daily_Commute_km      3498.4
Environmental_Concern_Level      1228.0
        Range_Anxiety_Level       603.2
     Home_Charging_Possible       358.6
          Subsidy_Available       343.8


Evaluating Option B (All Features with Subsampling 0.8) (13 features)
Features: ['Daily_Commute_km', 'Annual_Income_USD', 'Env

In [ ]:
test_path = 'test.csv'
test_df = pd.read_csv(test_path)

# encode the categoricals to match the training data format
for col in cat_cols:
    if col in test_df.columns:
        test_df[col] = test_df[col].astype('category')

# define the parameters for the final models
# (n_estimators is set to 400 based on the average best_iteration from CV to prevent overfitting)
final_params = {
    'objective': 'binary', 
    'metric': 'auc', 
    'boosting_type': 'gbdt',
    'learning_rate': 0.05, 
    'num_leaves': 31, 
    'subsample': 0.8, 
    'n_estimators': 400, 
    'random_state': 42, 
    'n_jobs': -1, 
    'verbose': -1
}

# train the final Model A (6 features) on the entire training dataset
final_model_a = lgb.LGBMClassifier(**final_params, colsample_bytree=1.0)
final_model_a.fit(data[strong_features], y)
test_preds_a = final_model_a.predict_proba(test_df[strong_features])[:, 1]

# train the final Model B (13 features) on the entire training dataset
final_model_b = lgb.LGBMClassifier(**final_params, colsample_bytree=0.8)
final_model_b.fit(data[all_features], y)
test_preds_b = final_model_b.predict_proba(test_df[all_features])[:, 1]

sub_a = pd.DataFrame({'id': test_df['id'], 'Will_Buy_EV': test_preds_a})
sub_a.to_csv('submission_model_a_6features.csv', index=False)

sub_b = pd.DataFrame({'id': test_df['id'], 'Will_Buy_EV': test_preds_b})
sub_b.to_csv('submission_model_b_13features.csv', index=False)

print('Model A\'s first five outputs:')
print(sub_a.head())
print('\n')
print('Model B\'s first five outputs:')
print(sub_b.head())

Model A's first five outputs:
       id  Will_Buy_EV
0  668665     0.009787
1  668666     0.018568
2  668667     0.005765
3  668668     0.002288
4  668669     0.016472


Model B's first five outputs:
       id  Will_Buy_EV
0  668665     0.008409
1  668666     0.015758
2  668667     0.005369
3  668668     0.002990
4  668669     0.021014


### 7. Final Conclusions

As demonstrated by the cross-validation results, Model A (6 Features) achieves nearly identical predictive performance (ROC-AUC ~0.941) compared to Model B (13 Features). The addition of 7 demographic and geographic variables provided a mathematically negligible lift (+0.0005) while increasing training time and complexity. Validated against the unseen Kaggle test dataset, both models achieved near-identical predictive performance, scoring an ROC-AUC of 0.94107 for Model A (6 features) and 0.94148 for Model B (13 features). This of 0.00041 can confirm that our cross-validation findings: incorporating extra demographic and infrastructure metrics yields no significant improvement in predictive power, validating the operational and economic superiority of the lean, 6-feature model.

Yao Yan, Walter Reade, Elizabeth Park. Predicting Electric Vehicle Purchases. https://kaggle.com/competitions/playground-series-s6e9, 2026. Kaggle.